# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [14]:
# imports
import os
from dotenv import load_dotenv
import requests
from scraper import fetch_website_contents
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

MODEL_GEMINI = 'gemini-2.5-flash-lite'
MODEL_LLAMA = 'llama3.2:1b'


In [9]:
# set up environment and get the right keys
google_api_key = os.getenv("GOOGLE_API_KEY")

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)


In [18]:
# here is the question; type over this to ask something new
question = """
What is the point of approximate bayesian computation. When would I use it in genomics analysis? How would you run the simulations in python?
"""
website = "https://en.wikipedia.org/wiki/Approximate_Bayesian_computation"
#"https://wandb.ai/byyoung3/Generative-AI/reports/Budget-forcing-s1-32B-Waiting-is-all-you-need---VmlldzoxMTIzNzkzOA"


In [ ]:
# Get gpt-4o-mini to answer, with streaming

system_prompt = "You are a helpful technical tutor who answers questions about python code, software engineering, data science and LLMs"

user_prompt = "Please give a detailed explanation to the following question. The answer is intended to be easily understood by your grandma. Reference material is also included" 

def get_ai_response_gemini(url, question): 
    ref_material = fetch_website_contents(url)
    stream = gemini.chat.completions.create(
        model=MODEL_GEMINI,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + ref_material + question}
          ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)    

In [19]:
# Get Llama 3.2 to answer
get_ai_response_gemini(website, question)

Let's break down Approximate Bayesian Computation (ABC) for you, just like we're talking about a recipe or a craft project. Don't worry about the fancy words; we'll make it easy to understand.

## What is the Point of Approximate Bayesian Computation (ABC)?

Imagine you have a complex recipe for a cake. You know all the ingredients (like flour, sugar, eggs), but the *exact proportions* that lead to the *perfect* cake are a bit mysterious. You want to figure out what those perfect proportions are, but you can't just taste every possible combination – that would take forever and be a waste of ingredients!

This is where ABC comes in. In a nutshell, **ABC is a way to make educated guesses about the unknown parts of a scientific model when it's too hard to calculate the exact answer.**

Think of it like this:

*   **The "Model":** This is your recipe. It describes how you think the cake is made (e.g., "if I add more sugar, the cake gets sweeter").
*   **The "Unknown Parts":** These are the exact amounts of each ingredient. We don't know them perfectly.
*   **The "Data":** This is the cake you actually baked. You can look at it, taste it, and see how it turned out (e.g., "this cake is a little too sweet," or "this cake has a great crumb").
*   **The "Goal":** You want to find the ingredient amounts that would produce a cake that looks and tastes exactly like the one you baked.

**The Problem:** With some "recipes" (models), especially in science and statistics, it's mathematically impossible or incredibly difficult to directly calculate the best ingredient amounts based on how the cake turned out. It's like trying to reverse-engineer the exact baking process just by tasting the final cake.

**The ABC Solution:** Instead of trying to calculate the perfect proportions directly, ABC does something clever:

1.  **It "Bakes" Many Cakes:** It randomly picks a set of ingredient amounts (these are its guesses for the unknown parts).
2.  **It "Bakes" Another Cake:** Using those guessed amounts, it simulates or "bakes" a new cake according to its recipe.
3.  **It Compares:** It then compares this *simulated* cake to the *real* cake you baked. Is it close enough?
4.  **It Keeps the Good Guesses:** If the simulated cake is *very similar* to the real cake (meaning the guessed ingredient amounts were pretty good), it keeps those ingredient amounts as a valid possibility.
5.  **It Discards the Bad Guesses:** If the simulated cake is very different from the real cake, it throws away those ingredient amounts as likely wrong.
6.  **Repeat, Repeat, Repeat:** It does this thousands or millions of times, each time picking new random ingredient amounts, simulating a cake, and comparing.

**The Result:** By collecting all the ingredient amounts that produced simulated cakes very close to the real one, you end up with a good set of *likely* proportions for your ingredients. It's an *approximation* because you're not finding the one single perfect answer, but a range of answers that are good enough.

**So, the point of ABC is to get a useful estimate of unknown quantities in a model when direct calculation is impossible or too hard, by simulating data based on guesses and keeping the guesses that produce data similar to what we observed.**

## When Would I Use It in Genomics Analysis?

Genomics is a perfect playground for ABC! Think about it: studying genes and DNA is incredibly complex.

Here are some scenarios in genomics where ABC is a lifesaver:

1.  **Evolutionary Models:**
    *   **Scenario:** Scientists want to understand how a species' DNA has changed over time. They have a model of how mutations happen, how populations split, and how genes flow between them. They have DNA samples from living individuals.
    *   **The Problem:** It's incredibly hard to calculate directly what the "evolutionary history" (like when populations split, how big they were) must have been to produce the DNA sequences they see today.
    *   **How ABC Helps:** They can use ABC to simulate many possible evolutionary histories. For each simulated history, they generate *synthetic DNA sequences* that *would have been produced* by that history. They then compare these synthetic sequences to the real DNA sequences they collected. If the synthetic sequences look very similar to the real ones, that simulated evolutionary history is considered a good explanation. This helps them estimate things like divergence times and population sizes.

2.  **Population Genetics:**
    *   **Scenario:** Researchers are studying how genetic variation exists within a population. They have a model describing gene flow, selection (which genes are more or less "successful"), and mutation. They have genetic data from many individuals.
    *   **The Problem:** Determining the exact values of parameters in these complex population genetics models (like the strength of selection or the rate of mutation) from the observed genetic data can be computationally challenging.
    *   **How ABC Helps:** They can simulate genetic data from populations with different parameter values. They then compare the simulated genetic patterns (like the distribution of allele frequencies) to the patterns observed in their real data. This allows them to estimate the parameters of the population genetics model.

3.  **Disease Association Studies (with complex models):**
    *   **Scenario:** When trying to understand how multiple genes contribute to a disease, the models can become very complicated. Instead of just one gene, you might be looking at how 10 or 20 genes interact.
    *   **The Problem:** Directly calculating the probability of seeing certain gene combinations in patients versus healthy people for a very complex gene interaction model is often not feasible.
    *   **How ABC Helps:** You can simulate individuals with different combinations of genetic risk factors and see what the expected disease outcomes would be. You compare these simulated outcomes to actual patient data to refine your understanding of which gene combinations are important.

Essentially, any time you have a complex scientific process you're trying to understand (like evolution or population dynamics), and you have data from that process, but the math to connect the underlying factors to the data is too tricky, ABC is a powerful tool. It's like playing a game of "what if" with a computer to find the most plausible explanations.

## How Would You Run the Simulations in Python?

Python is fantastic for ABC because it's great for programming, data handling, and has many scientific libraries. Here's a conceptual outline of how you'd do it, using a very simplified example.

Let's say we're trying to figure out the average height of a group of people. We have a *model* that says heights follow a "normal distribution" (a bell curve). We don't know the exact average height ($\mu$) or how spread out the heights are ($\sigma$). We have some observed heights (our "data").

**Conceptual Python Code for ABC:**

```python
import numpy as np
import scipy.stats as stats
import random

# --- 1. Our Observed Data ---
# Imagine these are the heights we measured in our real sample
observed_heights = np.array([170, 175, 168, 180, 172, 178, 165, 173, 176, 171])

# --- 2. Our "Model" (the function that generates data) ---
# This function simulates heights based on a given average (mu) and spread (sigma)
def simulate_heights(mu, sigma, num_samples):
    return np.random.normal(loc=mu, scale=sigma, size=num_samples)

# --- 3. A Way to "Compare" Simulated Data to Observed Data ---
# This is our "summary statistic" or "distance metric".
# A simple one is the mean of the simulated heights vs. the mean of observed heights.
# We want this difference to be small.
def compare_data(simulated_data, observed_data):
    mean_diff = np.abs(np.mean(simulated_data) - np.mean(observed_data))
    return mean_diff

# --- 4. The ABC Rejection Algorithm ---
def abc_rejection_sampler(observed_data, num_simulations, tolerance):
    accepted_params = [] # To store the (mu, sigma) pairs that are "good"

    # We need to guess a range for our unknown parameters (mu and sigma)
    # Let's assume average height is likely between 160 and 190, and spread between 5 and 15
    mu_range = (160, 190)
    sigma_range = (5, 15)

    for _ in range(num_simulations):
        # a. Randomly pick "guess" parameters
        # In a real scenario, we might have prior distributions for these.
        # For simplicity, we're just sampling uniformly from a range.
        guessed_mu = random.uniform(mu_range[0], mu_range[1])
        guessed_sigma = random.uniform(sigma_range[0], sigma_range[1])
        current_params = (guessed_mu, guessed_sigma)

        # b. Simulate data using the guessed parameters
        num_observed_samples = len(observed_data) # Simulate same number of samples
        simulated_data = simulate_heights(guessed_mu, guessed_sigma, num_observed_samples)

        # c. Compare simulated data to observed data
        distance = compare_data(simulated_data, observed_data)

        # d. If the simulated data is close enough to the observed data, accept the parameters
        if distance < tolerance:
            accepted_params.append(current_params)

    return accepted_params

# --- How to Run It ---
num_simulations_to_run = 10000  # How many times we'll try to bake a cake
tolerance_level = 2.0          # How "close" the simulated cake needs to be to the real one
                               # (This is crucial and often needs tuning!)

print("Running ABC simulations...")
good_params = abc_rejection_sampler(observed_heights, num_simulations_to_run, tolerance_level)

print(f"Accepted {len(good_params)} parameter sets out of {num_simulations_to_run}.")

# --- Analyze the Accepted Parameters ---
if good_params:
    # Convert list of tuples to a numpy array for easier analysis
    accepted_params_array = np.array(good_params)

    # Estimate the posterior distribution of our parameters
    # We can look at the average of the accepted mu and sigma, or their distributions
    estimated_mu = np.mean(accepted_params_array[:, 0])
    estimated_sigma = np.mean(accepted_params_array[:, 1])

    print(f"\nEstimated average height (mu) based on ABC: {estimated_mu:.2f}")
    print(f"Estimated spread (sigma) based on ABC: {estimated_sigma:.2f}")

    # You could also plot histograms of accepted_params_array[:, 0] and accepted_params_array[:, 1]
    # to see the range of plausible values.
else:
    print("No parameter sets were accepted. Try increasing num_simulations or tolerance.")

```

**Explanation for Grandma:**

1.  **`observed_heights`**: This is like the list of actual heights we measured from a group of people. This is our "real cake."
2.  **`simulate_heights(mu, sigma, num_samples)`**: This is our "recipe." It takes a guess for the *average height* (`mu`) and how much the heights *vary* (`sigma`), and then it *makes up* a list of heights that would be expected if those were the true average and variation. It's like baking a pretend cake with specific ingredients.
3.  **`compare_data(simulated_data, observed_data)`**: This is how we "taste" and compare. Here, we're just checking if the *average height* of our pretend cake is close to the *average height* of our real cake. The smaller the difference, the better our pretend cake is. In real science, this comparison can be much more sophisticated.
4.  **`abc_rejection_sampler(...)`**: This is the main process.
    *   It runs many times (`num_simulations`).
    *   In each run, it first **guesses** what the average height (`guessed_mu`) and variation (`guessed_sigma`) might be. (In the code, it picks them randomly from plausible ranges).
    *   Then, it uses our "recipe" (`simulate_heights`) to **bake a pretend cake** with those guessed ingredients.
    *   It then **compares** the pretend cake to the real cake using `compare_data`.
    *   If the pretend cake is **close enough** to the real cake (based on our `tolerance_level`), it keeps those guessed ingredients (`guessed_mu`, `guessed_sigma`) as potentially correct.
    *   If it's not close enough, it throws those guesses away.
5.  **Analyzing Results**: After many tries, we look at all the ingredient guesses that resulted in a "good enough" pretend cake. The average of these good guesses gives us our best estimate for the true average height and variation in the original group of people.

**Important Notes for Python:**

*   **Libraries:** We use `numpy` for numerical operations (like calculating averages), `scipy.stats` for statistical functions (though `np.random.normal` is enough here), and `random` for picking random numbers.
*   **`tolerance`:** This is the most critical part of ABC. A very small tolerance means you're only accepting simulations that are *extremely* close, which might lead to few accepted parameters. A large tolerance accepts simulations that are less precise, giving you more accepted parameters but a less accurate approximation. Finding the right balance is key.
*   **Summary Statistics:** The `compare_data` function is where you define your "summary statistics." In genomics, you wouldn't just compare averages. You might compare allele frequencies, genetic diversity measures, or patterns of linkage disequilibrium. The choice of summary statistics is crucial for ABC to work well.
*   **More Advanced ABC:** The code above shows the basic "rejection sampling" method. There are more sophisticated methods (like ABC-SMC) that are more efficient and can handle very complex models, but the core idea of simulating and comparing remains the same.

I hope this detailed explanation, with the cake analogy and Python example, makes Approximate Bayesian Computation much clearer! It's a powerful technique that helps scientists make sense of complex data when direct mathematical solutions are out of reach.